# 01 — K-means и его модификации

# Кластеризация данных: практическая серия ноутбуков

Cначала формулируется идея метода, затем показывается его геометрический смысл, визуализация и практическая реализация в Python. В исходном уроке акцент сделан на объяснении метода через визуальные представления и двумерные проекции;

> Кластеризация — обучение без учителя. Алгоритм не получает правильные классы объектов заранее, а пытается обнаружить естественную структуру данных.

## 1. Что такое K-means

**K-means** разбивает наблюдения на заранее заданное число `K` групп. Для каждого кластера алгоритм хранит центроид — среднее положение его точек.

Целевая функция — минимизация внутрикластерной суммы квадратов расстояний:

$$
J = \sum_{k=1}^{K}\sum_{x_i \in C_k} ||x_i-\mu_k||^2
$$

где $C_k$ — кластер, а $\mu_k$ — его центроид.

### Пошагово

1. Выбираем `K` начальных центроидов.
2. Каждую точку относим к ближайшему центроиду.
3. Для каждого кластера пересчитываем центроид как среднее его точек.
4. Повторяем шаги 2–3 до сходимости.
5. Получаем метку кластера для каждой строки.

K-means хорошо работает, когда группы примерно компактные, выпуклые и разделены расстоянием. Он чувствителен к масштабу признаков, выбросам и выбору `K`.

## 2. Преимущества и недостатки

**Преимущества**
- простой для понимания и реализации;
- хорошо масштабируется по числу объектов;
- центроиды легко интерпретировать;
- хорошо подходит для компактных кластеров.

**Недостатки**
- число кластеров `K` нужно выбрать заранее;
- чувствителен к выбросам;
- результат зависит от начальной инициализации;
- предполагает геометрию, близкую к сферическим/выпуклым кластерам;
- расстояния имеют смысл только при корректном масштабе признаков.

### Сколько кластеров выбрать?

Единственного универсального ответа нет. Обычно проверяют несколько значений `K` и используют:
- **elbow method** — ищем точку, после которой уменьшение inertia становится небольшим;
- **silhouette score** — насколько точки близки к своему кластеру и отделены от соседних;
- доменную интерпретируемость;
- устойчивость кластеров при повторных запусках.

Поэтому `K=3` или `K=4` нельзя объявлять правильным только потому, что алгоритм это позволил.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mglearn

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# Учебный набор с тремя компактными группами
X, y_true = make_blobs(
    n_samples=600,
    centers=3,
    cluster_std=[1.0, 1.4, 0.8],
    random_state=42
)

X_scaled = StandardScaler().fit_transform(X)

## 3. Базовый K-means: визуализация результата

Ниже каждая точка — наблюдение, цвет — найденный кластер. Для двух признаков такой график показывает геометрический смысл алгоритма непосредственно.

In [ ]:
kmeans = KMeans(n_clusters=3, init="random", n_init=20, random_state=42)
labels = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(9, 6))
mglearn.discrete_scatter(
    X_scaled[:, 0], X_scaled[:, 1], 
    labels, ax=plt.gca(), s=25)
plt.scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    marker="X", s=220, edgecolor="black"
)
plt.title("K-means: найденные кластеры и центроиды")
plt.xlabel("Признак 1 (после стандартизации)")
plt.ylabel("Признак 2 (после стандартизации)")
plt.show()

**Что показывает график:** K-means формирует области вокруг центроидов. Для компактных облаков это естественное разбиение. Если бы данные имели форму колец, дуг или сильно отличающуюся плотность, такой подход мог бы дать неудовлетворительный результат.

## 4. Выбор K: inertia и silhouette

`inertia_` — значение целевой функции K-means. Оно **всегда не возрастает** при увеличении `K`, поэтому нельзя просто выбрать минимальное значение: при `K=n` каждая точка станет собственным кластером.

Silhouette находится примерно в диапазоне от `-1` до `1`; чем выше, тем лучше разделены найденные группы.

In [ ]:
rows = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=20, random_state=42)
    lab = model.fit_predict(X_scaled)
    rows.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X_scaled, lab)
    })

selection = pd.DataFrame(rows)
display(selection)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].plot(selection["k"], selection["inertia"], marker="o")
ax[0].set_title("Elbow method: inertia")
ax[0].set_xlabel("Количество кластеров K")
ax[0].set_ylabel("Inertia")

ax[1].plot(selection["k"], selection["silhouette"], marker="o")
ax[1].set_title("Silhouette score")
ax[1].set_xlabel("Количество кластеров K")
ax[1].set_ylabel("Silhouette")

plt.tight_layout()
plt.show()

## 5. Почему появился K-means++?

Обычная случайная инициализация может поставить несколько центроидов слишком близко. Тогда алгоритм начинает с неудачной конфигурации и может попасть в локальный минимум.

**K-means++** выбирает начальные центроиды так, чтобы новые центры с большей вероятностью находились далеко от уже выбранных. Это обычно даёт более удачную стартовую конфигурацию.

В `scikit-learn` K-means++ включается параметром `init="k-means++"` и используется по умолчанию.

In [ ]:
random_model = KMeans(
    n_clusters=3, init="random", n_init=1, random_state=7
).fit(X_scaled)

plus_model = KMeans(
    n_clusters=3, init="k-means++", n_init=1, random_state=7
).fit(X_scaled)

comparison = pd.DataFrame({
    "method": ["random", "k-means++"],
    "inertia": [random_model.inertia_, plus_model.inertia_],
    "silhouette": [
        silhouette_score(X_scaled, random_model.labels_),
        silhouette_score(X_scaled, plus_model.labels_)
    ]
})
display(comparison)

## 6. Mini-batch K-means

**MiniBatchKMeans** обновляет центроиды не по всему датасету, а по небольшим случайным пакетам (`batch_size`). Это уменьшает вычислительную нагрузку на очень больших данных.

**Плюсы:** быстрее и экономнее по памяти на больших наборах.

**Минусы:** результат может быть немного хуже обычного K-means; качество зависит от размера batch.

Использовать его стоит, когда обычный K-means становится дорогим, а небольшая потеря качества допустима.

In [ ]:
mb = MiniBatchKMeans(
    n_clusters=3,
    batch_size=64,
    n_init=10,
    random_state=42
)
mb_labels = mb.fit_predict(X_scaled)

display(pd.DataFrame({
    "algorithm": ["KMeans", "MiniBatchKMeans"],
    "inertia": [kmeans.inertia_, mb.inertia_],
    "silhouette": [
        silhouette_score(X_scaled, labels),
        silhouette_score(X_scaled, mb_labels)
    ]
}))

## 7. Псевдоалгоритм K-means на Python

Это **не промышленная реализация**, а учебный код, показывающий внутреннюю логику метода.

In [ ]:
# Псевдоалгоритм / частичная реализация

def simple_kmeans(X, k, max_iter=100):
    # 1. Случайно выбираем k центроидов
    centers = X[np.random.choice(len(X), k, replace=False)]

    for _ in range(max_iter):
        # 2. Расстояние каждой точки до каждого центра
        distances = np.linalg.norm(
            X[:, None, :] - centers[None, :, :],
            axis=2
        )

        # 3. Назначаем ближайший кластер
        labels = distances.argmin(axis=1)

        # 4. Пересчитываем центры
        new_centers = np.array([
            X[labels == j].mean(axis=0) if np.any(labels == j)
            else centers[j]
            for j in range(k)
        ])

        # 5. Проверяем сходимость
        if np.allclose(centers, new_centers):
            break

        centers = new_centers

    return labels, centers

## 8. Итог

| Вариант | Главная идея | Когда полезен |
|---|---|---|
| K-means | классические итерации по всем точкам | базовый случай |
| K-means++ | более разумная инициализация | почти всегда хороший default |
| MiniBatchKMeans | обучение на mini-batch | очень большие датасеты |

Главный вопрос K-means — не только «как запустить алгоритм», но и **насколько выбранное `K` соответствует структуре данных и бизнес-смыслу**.